# AyyPhoto — tester un modèle sauvegardé sur une photo

Ce notebook est **indépendant de l'entraînement et de DIV2K**. Il charge un
modèle déjà entraîné, traite une photo JPEG/PNG et enregistre une nouvelle image.
Il ne lance aucun entraînement, ne monte pas Drive et ne télécharge pas de dataset.
Le CPU fonctionne aussi si aucun GPU n'est disponible, avec une vitesse différente.

## Kaggle : démarrage

1. Importer ce notebook dans Kaggle.
2. Via **Add Input**, attacher les sorties de l'ancien notebook ou un dataset
   contenant le fichier sauvegardé : `model_bundle.pt`, `best_weights.pth` ou
   `last.ckpt`. Une archive `*_export.zip` AyyPhoto est aussi acceptée.
3. Ajouter une photo à tester et copier son chemin dans `PHOTO_PATH`.
4. Renseigner `MODEL_PATH` si plusieurs expériences sont attachées, puis **Run All**.

Les anciens poids du CNN AyyPhoto sont compatibles : l'architecture est déduite
des formes des poids et vérifiée avant chargement. Aucune image d'entraînement
n'est nécessaire. Si les poids ont été perdus lors d'une ancienne session et
qu'aucune sauvegarde n'existe, ce notebook ne peut pas les reconstruire.

Les sorties sont dans `/kaggle/working/ayyphoto_tests`. Conserver ou télécharger
les résultats avant de quitter la session. Sans photo de référence propre
alignée, aucun PSNR/SSIM n'est annoncé comme mesure de qualité.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    packages = ["numpy>=1.26,<3", "Pillow>=10,<13", "matplotlib>=3.8,<4", "tqdm>=4.66,<5"]
    if importlib.util.find_spec("torch") is None:
        packages.append("torch>=2.3,<3")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

MODEL_PATH = ""  # Chemin .pt/.pth/.ckpt ou *_export.zip ; vide = détection des poids.
PHOTO_PATH = ""  # Obligatoire : ex. "/kaggle/input/mes-photos/photo.jpg".
OUTPUT_DIR = Path("/kaggle/working/ayyphoto_tests")
TILE_SIZE = 256
STRENGTH = 1.0  # 0 = original ; 1 = sortie du modèle ; valeurs intermédiaires = mélange.
assert TILE_SIZE > 0 and 0 <= STRENGTH <= 1


In [ ]:
import hashlib
import io
import json
import re
import time
import zipfile
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageCms, ImageOps
from torch import nn
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Périphérique :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))


## Trouver et charger les poids

Dans chaque dossier, la recherche privilégie `model_bundle.pt`, puis
`best_weights.pth`, `last.ckpt` et `interrupted_weights.pth`. Si plusieurs
expériences sont présentes, choisir explicitement `MODEL_PATH`.
Une archive ZIP est lue sans extraction de fichiers sur disque. Les poids
interrompus ou `last.ckpt` peuvent différer du meilleur modèle de validation.


In [ ]:
def discover_model_files(roots):
    preferred = ("model_bundle.pt", "best_weights.pth", "last.ckpt", "interrupted_weights.pth")
    by_directory, archives = {}, set()
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for rank, name in enumerate(preferred):
            for file in root.rglob(name):
                if not file.is_file():
                    continue
                file = file.resolve()
                previous = by_directory.get(file.parent)
                if previous is None or rank < previous[0]:
                    by_directory[file.parent] = (rank, file)
        archives.update(file.resolve() for file in root.rglob("*_export.zip") if file.is_file())
    models = sorted(value[1] for value in by_directory.values())
    # Les ZIP servent de solution de repli si aucun poids extrait n'est trouvé.
    return models if models else sorted(archives)

def read_model_payload(path):
    path = Path(path)
    limit = 256 * 1024 * 1024  # Suffisant pour ce petit CNN et ses checkpoints.
    if not path.is_file():
        raise FileNotFoundError(f"Fichier du modèle introuvable : {path}")
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            entries = [entry for entry in archive.infolist() if not entry.is_dir()]
            priority = {name: rank for rank, name in enumerate(
                ("model_bundle.pt", "best_weights.pth", "last.ckpt", "interrupted_weights.pth"))}
            candidates = [entry for entry in entries if Path(entry.filename).name in priority]
            if not candidates:
                raise ValueError("L’archive ne contient aucun poids AyyPhoto reconnu.")
            top_rank = min(priority[Path(entry.filename).name] for entry in candidates)
            candidates = [entry for entry in candidates if priority[Path(entry.filename).name] == top_rank]
            if len(candidates) != 1:
                raise ValueError("Plusieurs modèles dans l’archive : fournir un fichier de poids précis.")
            entry = candidates[0]
            if entry.file_size > limit:
                raise ValueError("Poids trop volumineux pour le CNN prévu dans ce notebook.")
            raw = archive.read(entry)
            provenance = f"{path}!{entry.filename}"
    else:
        if path.stat().st_size > limit:
            raise ValueError("Fichier trop volumineux pour le CNN prévu dans ce notebook.")
        raw, provenance = path.read_bytes(), str(path)
    return torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True), provenance, hashlib.sha256(raw).hexdigest()

def infer_architecture(weights):
    convolution_indices = sorted(int(match.group(1)) for key in weights
        if (match := re.fullmatch(r"network\.(\d+)\.weight", key)))
    depth = len(convolution_indices)
    if depth < 2 or convolution_indices != list(range(0, 2 * depth, 2)):
        raise ValueError("Ces poids ne correspondent pas au CNN résiduel AyyPhoto.")
    channels = int(weights["network.0.weight"].shape[0])
    if channels <= 0:
        raise ValueError("Nombre de canaux invalide.")
    for position, index in enumerate(convolution_indices):
        expected = (channels, 3, 3, 3) if position == 0 else (
            (3, channels, 3, 3) if position == depth - 1 else (channels, channels, 3, 3))
        if tuple(weights[f"network.{index}.weight"].shape) != expected:
            raise ValueError(f"Forme de convolution incompatible à la couche {index}.")
    return {"channels": channels, "depth": depth}

if not MODEL_PATH:
    found = discover_model_files([Path("/kaggle/input"), Path("/kaggle/working")])
    if len(found) != 1:
        print("Fichiers candidats :", *map(str, found[:20]), sep="\n")
        raise ValueError("Renseigner MODEL_PATH avec les poids conservés après l’entraînement.")
    MODEL_PATH = str(found[0])

payload, MODEL_SOURCE, MODEL_SHA256 = read_model_payload(MODEL_PATH)
if not isinstance(payload, dict):
    raise ValueError("Format du fichier de poids non reconnu.")
if "architecture" in payload and payload["architecture"] != "residual_rgb_cnn":
    raise ValueError("Cette architecture n’est pas prise en charge par ce notebook.")
weights = payload.get("model_state_dict", payload)
if not isinstance(weights, dict) or not weights:
    raise ValueError("Poids absents du fichier.")
if all(key.startswith("module.") for key in weights):
    weights = {key.removeprefix("module."): value for key, value in weights.items()}
MODEL_CFG = infer_architecture(weights)
metadata_cfg = payload.get("config", {})
for key in ("channels", "depth"):
    if key in metadata_cfg and metadata_cfg[key] != MODEL_CFG[key]:
        raise ValueError("Les métadonnées d’architecture ne correspondent pas aux poids.")
print("Poids :", MODEL_SOURCE)
print("Architecture :", MODEL_CFG)


In [ ]:
class AyyPhotoDenoiser(nn.Module):
    def __init__(self, channels=48, depth=8):
        super().__init__()
        layers = [nn.Conv2d(3, channels, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(channels, 3, 3, padding=1)]
        self.network = nn.Sequential(*layers)
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, image):
        return image - self.network(image)


model = AyyPhotoDenoiser(**MODEL_CFG).to(DEVICE)
for name, value in weights.items():
    if not torch.is_tensor(value) or not torch.isfinite(value).all():
        raise ValueError(f"Poids invalides ou non finis : {name}")
model.load_state_dict(weights, strict=True)
model.eval()
print("Modèle chargé ; aucun entraînement nécessaire.")


## Traiter la photo

Le prétraitement est identique à celui de l’entraînement : orientation EXIF,
conversion ICC vers sRGB puis RGB `[0, 1]`. L’image est traitée par tuiles avec
une marge correspondant au champ réceptif du CNN. L’original n’est pas écrasé.
`STRENGTH` permet de réduire visuellement l’intensité si le résultat est trop lissé ;
cela ne constitue pas une amélioration mesurée ni une garantie de fidélité.


In [ ]:
def load_rgb(path):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source)
        if image.mode in ("RGBA", "LA") or "transparency" in image.info:
            raise ValueError("Image avec transparence : fournir une photo RGB opaque.")
        profile = image.info.get("icc_profile")
        if profile:
            image = ImageCms.profileToProfile(
                image, ImageCms.ImageCmsProfile(io.BytesIO(profile)),
                ImageCms.createProfile("sRGB"), outputMode="RGB",
            )
        else:
            image = image.convert("RGB")
        return np.array(image, dtype=np.uint8, copy=True)


@torch.inference_mode()
def restore_photo(rgb, tile_size, halo):
    tensor = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0
    _, height, width = tensor.shape
    result = torch.empty_like(tensor)
    for top in tqdm(range(0, height, tile_size), desc="Débruitage"):
        bottom = min(top + tile_size, height)
        for left in range(0, width, tile_size):
            right = min(left + tile_size, width)
            y0, y1 = max(0, top - halo), min(height, bottom + halo)
            x0, x1 = max(0, left - halo), min(width, right + halo)
            patch = tensor[:, y0:y1, x0:x1].unsqueeze(0).to(DEVICE)
            restored = model(patch)[0].clamp(0, 1).cpu()
            if not torch.isfinite(restored).all():
                raise FloatingPointError("Le modèle a produit des pixels non finis ; aucune image n'est enregistrée.")
            result[:, top:bottom, left:right] = restored[:, top - y0:bottom - y0, left - x0:right - x0]
    mixed = tensor * (1 - STRENGTH) + result * STRENGTH
    return np.rint(mixed.permute(1, 2, 0).numpy().clip(0, 1) * 255).astype(np.uint8)

if not PHOTO_PATH:
    raise ValueError("Renseigner PHOTO_PATH avec le chemin d’une photo JPEG/PNG attachée au notebook.")
photo_path = Path(PHOTO_PATH).expanduser().resolve()
original = load_rgb(photo_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_path = OUTPUT_DIR / f"{photo_path.stem}_ayyphoto_{stamp}.png"
if output_path.resolve() == photo_path:
    raise ValueError("Le fichier de sortie doit être différent de l’original.")
started = time.perf_counter()
enhanced = restore_photo(original, TILE_SIZE, halo=max(16, MODEL_CFG["depth"]))
seconds = time.perf_counter() - started
profile = ImageCms.ImageCmsProfile(ImageCms.createProfile("sRGB")).tobytes()
Image.fromarray(enhanced).save(output_path, icc_profile=profile)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, photo, title in zip(axes, (original, enhanced), ("Original", "AyyPhoto — résultat")):
    ax.imshow(photo)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(output_path.with_name(output_path.stem + "_comparison.png"), dpi=140)
plt.show()
report = {
    "model_source": MODEL_SOURCE, "model_sha256": MODEL_SHA256,
    "architecture": MODEL_CFG, "device": str(DEVICE),
    "strength": STRENGTH, "tile_size": TILE_SIZE,
    "width": int(enhanced.shape[1]), "height": int(enhanced.shape[0]),
    "inference_wall_seconds": seconds,
    "output": str(output_path),
    "quality_metrics": None,
    "note": "No aligned clean reference: inspect visually; no PSNR/SSIM quality claim.",
}
output_path.with_suffix(".json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Image corrigée :", output_path)
print("Rapport :", output_path.with_suffix(".json"))


## Après le test

Inspecter à 100 % les textures, les visages, le texte et les couleurs. Comparer
plusieurs photos ; une image synthétique réussie ne valide pas toutes les caméras.
Télécharger le PNG et son rapport ou conserver une version Kaggle avec les sorties.

- [Chargement des modèles PyTorch](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)
- [Notebooks Kaggle](https://www.kaggle.com/docs/notebooks)
